# Step 3 — Charging Infrastructure and Uncontrolled Charging

This step introduces the group's **designed** fleet and chargers. The course supplies depot load profiles and operating descriptions; fleet size/mix, daily vehicle energy and charger sizing are engineering assumptions that must be declared and justified.

### What this cell does — Load fleet design and physical network

Loads the explicit assumption table and charger design generated by preprocessing. The physical network remains the Step-2 all-closed, half-original-load network at the shared bus.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config
from src.data import load_fleet, load_fleet_assumptions, load_charger_design, load_base_loads, load_bus_mapping, installed_charger_power, load_prices
from src.fleet.uncontrolled import build_uncontrolled_schedule, charging_summary
from src.network.model import build_network
from src.network.timeseries import run_timeseries, check_installed_chargers_feasibility, calculate_shared_bus_critical_headroom, build_snapshot_network
from src.network.contingency import run_n1, SWITCH_CONFIGS_ALL_CLOSED
from src.reporting.metrics import scenario_metrics
from src.reporting.plots import save_schedule_plot, save_network_plot

fleet = load_fleet()
assumptions = load_fleet_assumptions()
charger_design = load_charger_design()
base_p, base_q = load_base_loads()
bus_map = load_bus_mapping(require_shared=True)
charger_power = installed_charger_power(fleet)
prices = load_prices()
network_factory = lambda: build_network(halve_existing_loads=True, close_ring_switches=True)

display(assumptions)
display(charger_design)
print("Installed charging power [MW]:", charger_power)


### What this cell does — Uncontrolled charging

Builds the reference 'dumb charging' schedule: whenever vehicles are available, charging starts immediately at installed depot power until that day's required battery energy is delivered. This is the baseline against which both optimizations are compared.

In [ ]:
schedule = build_uncontrolled_schedule(fleet)
charge_summary = charging_summary(schedule)
display(charge_summary)


### What this cell does — Validate the charging infrastructure and create Step-4 network limits

At each of the 336 hours, all installed chargers are switched to full power simultaneously on top of the Step-2 depot loads. If all hours remain within voltage, line and transformer limits, the existing charger ratings themselves are safe optimization bounds. This file is the network constraint interface used by Steps 4 and 5.

In [ ]:
network_limits = check_installed_chargers_feasibility(network_factory, base_p, base_q, bus_map, charger_power)
config.NETWORK_DIR.mkdir(parents=True, exist_ok=True)
network_limits.to_csv(config.NETWORK_LIMITS, index=False)
hour_ok = network_limits.groupby("time").all_installed_feasible.first()
print("Full installed charging feasible hours:", int(hour_ok.sum()), "/", len(hour_ok))
if not bool(hour_ok.all()):
    print("WARNING: Charger design is not feasible for every hour; revise design/connection point before Step 4.")


### What this cell does — AC validation and N-1 at peak uncontrolled charging

Runs the actual uncontrolled schedule through 336 AC power flows. The contingency snapshot is selected at the hour of **maximum uncontrolled EV charging**, matching the project instruction to test peak uncontrolled charging. Both normal and strict voltage limits are evaluated with all switches closed.

In [ ]:
results = run_timeseries(network_factory, base_p, base_q, schedule, bus_map)
peak_time = results.loc[results.total_ev_charging_MW.idxmax(), "time"]
peak_net = build_snapshot_network(network_factory, base_p, base_q, schedule, bus_map, peak_time)
n1_normal, n1_normal_viol = run_n1(peak_net, switch_configs=SWITCH_CONFIGS_ALL_CLOSED)
n1_strict, n1_strict_viol = run_n1(peak_net, v_min=config.V_MIN_STRICT, v_max=config.V_MAX_STRICT, switch_configs=SWITCH_CONFIGS_ALL_CLOSED)
metrics = scenario_metrics("uncontrolled", schedule, prices, results, n1_normal)
print("Peak uncontrolled charging hour:", peak_time)
display(metrics)


### What this cell does — Optional physical headroom diagnostic

Calculates aggregate extra EV hosting headroom only at the peak uncontrolled-charging hour. This is an engineering diagnostic, not the optimization constraint; the installed charger design is the relevant Step-4 bound if it is already fully network-feasible.

In [ ]:
headroom = calculate_shared_bus_critical_headroom(network_factory, base_p, base_q, bus_map, [peak_time], initial_high=sum(charger_power.values()))
display(headroom)


### What this cell does — Export Step 3

Writes the charger design evidence, uncontrolled schedule, physical results, network-limit interface and contingency tables.

In [ ]:
out = config.RESULTS / "step3_uncontrolled"
out.mkdir(parents=True, exist_ok=True)
schedule.to_csv(out / "uncontrolled_schedule.csv", index=False)
charge_summary.to_csv(out / "charging_summary.csv", index=False)
network_limits.to_csv(out / "network_limits.csv", index=False)
headroom.to_csv(out / "critical_headroom.csv", index=False)
results.to_csv(out / "network_results.csv", index=False)
metrics.to_csv(out / "scenario_metrics.csv", index=False)
n1_normal.to_csv(out / "n1_summary.csv", index=False)
n1_normal_viol.to_csv(out / "n1_violations.csv", index=False)
n1_strict.to_csv(out / "n1_strict_summary.csv", index=False)
n1_strict_viol.to_csv(out / "n1_strict_violations.csv", index=False)
save_schedule_plot(schedule, out / "uncontrolled_schedule.png", "Step 3 — Uncontrolled charging")
save_network_plot(results, out / "network_loading.png", "Step 3 — Network loading")
print("Saved to", out)
